# Notebook Setup

## Setting Directories

In [1]:
import sys
import os
from pathlib import Path

# Path to your project root
project_dir = Path(r"C:\Users\dominik.mika\dp100\dp100-learn")

# Change the working directory
os.chdir(project_dir)

# Add to sys.path if not already there
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))


# Now you can import from utils
from utils.consts import *

## Imports

### Local Imports

In [2]:
# Now you can import from utils
from utils.consts import SUBSCRIPTION_ID, PREFERED_RESOURCE_LOCATION, MAIN_STORAGE_ACCOUNT_ACCESS_KEY

### General Imports

In [3]:
import numpy as np
import pandas as pd

### Azure Imports

In [4]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

## Consts

In [5]:
subscription_id = SUBSCRIPTION_ID
azure_credentials = DefaultAzureCredential()

## Other

In [6]:
resource_group_name = "ml-workspace-dev"
resource_group_location = PREFERED_RESOURCE_LOCATION

In [7]:
storage_account_name = "dmpdp100storageaccount99"  # must be globally unique
storage_account_location = PREFERED_RESOURCE_LOCATION
storage_container_name = "dmpdp100data"
storage_account_access_key = MAIN_STORAGE_ACCOUNT_ACCESS_KEY

In [8]:
azureml_workspace_name = "mlw-dp100-labs"
azureml_resource_location = PREFERED_RESOURCE_LOCATION

In [9]:
datastore_name = "dmdp100datastore"

In [10]:
ml_client = MLClient(
    credential=azure_credentials,
    subscription_id=subscription_id,
    resource_group_name=resource_group_name,
    workspace_name=azureml_workspace_name
)

Failed to receive Azure VM metadata: [WinError 10054] An existing connection was forcibly closed by the remote host
Traceback (most recent call last):
  File "c:\Users\dominik.mika\dp100\dp100-learn\venv\Lib\site-packages\opentelemetry\resource\detector\azure\vm.py", line 66, in _get_azure_vm_metadata
    with urlopen(request, timeout=0.2) as response:
         ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dominik.mika\AppData\Local\Programs\Python\Python313\Lib\urllib\request.py", line 189, in urlopen
    return opener.open(url, data, timeout)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dominik.mika\AppData\Local\Programs\Python\Python313\Lib\urllib\request.py", line 489, in open
    response = self._open(req, data)
  File "C:\Users\dominik.mika\AppData\Local\Programs\Python\Python313\Lib\urllib\request.py", line 506, in _open
    result = self._call_chain(self.handle_open, protocol, protocol +
                              '_open', req)
  File "C:\Users\dominik.mika\

# Setup a Resource Group

In [19]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.resource import ResourceManagementClient

# Create resource management client
resource_client = ResourceManagementClient(azure_credentials, subscription_id)
try:
    # Try to get existing resource group
    rg_result = resource_client.resource_groups.get(resource_group_name)
    print(f"Resource group '{rg_result.name}' already exists in region '{rg_result.location}'")
except ResourceNotFoundError:
    # Create resource group if it doesn't exist
    rg_result = resource_client.resource_groups.create_or_update(
        resource_group_name,
        {"location": resource_group_location}
    )
    print(f"Provisioned resource group '{rg_result.name}' in the {rg_result.location} region")


# Optional lines to delete the resource group. begin_delete is asynchronous.
# poller = resource_client.resource_groups.begin_delete(rg_result.name)
# result = poller.result()

Resource group 'ml-workspace-dev' already exists in region 'westeurope'


# Setup Storage Account

## Create a Storage Account

In [9]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.storage import StorageManagementClient

storage_client = StorageManagementClient(azure_credentials, subscription_id)

try:
    # Try to get existing storage account
    storage_account = storage_client.storage_accounts.get_properties(
        resource_group_name=resource_group_name,
        account_name=storage_account_name
    )
    print(f"Storage account already exists: {storage_account.name}")
except ResourceNotFoundError:
    # If not found, create new storage account
    print("Creating storage account...")
    poller = storage_client.storage_accounts.begin_create(
        resource_group_name=resource_group_name,
        account_name=storage_account_name,
        parameters={
            "location": storage_account_location,
            "sku": {"name": "Standard_LRS"},
            "kind": "StorageV2",
            "enable_https_traffic_only": True
        }
    )
    storage_account = poller.result()
    print(f"Storage account created: {storage_account.name}")

# Extract the resource ID
storage_resource_id = storage_account.id
print(f"Storage Resource ID: {storage_resource_id}")

Storage account already exists: dmpdp100storageaccount99
Storage Resource ID: /subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.Storage/storageAccounts/dmpdp100storageaccount99


## Create Data Container

In [12]:
from azure.storage.blob import BlobServiceClient

# Build connection string
connection_string = (
    f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};"
    f"AccountKey={storage_account_access_key};EndpointSuffix=core.windows.net"
)

# Create blob service client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create container (if not exists)
try:
    blob_service_client.create_container(storage_container_name)
    print(f"Container '{storage_container_name}' created.")
except Exception as e:
    if "ContainerAlreadyExists" in str(e):
        print(f"Container '{storage_container_name}' already exists.")
    else:
        raise

Container 'dmpdp100data' already exists.


# Setup Azure ML Workspace

## Creating the Workspace

In [ ]:
from azure.core.exceptions import ResourceNotFoundError
from azure.ai.ml.entities import Workspace

try:
    # Try to get existing workspace
    ws = ml_client.workspaces.get(azureml_workspace_name)
    print(f"AML workspace already exists: {ws.name}")
except ResourceNotFoundError:
    # Create new AML workspace
    ws = Workspace(
        name=azureml_workspace_name,
        location=azureml_resource_location,
        resource_group_name=resource_group_name,
        storage_account=storage_resource_id
    )
    print("Creating AML workspace...")
    ml_client.workspaces.begin_create(ws).result()
    print(f"Workspace '{azureml_workspace_name}' created with default storage: {storage_account_name}")


Failed to receive Azure VM metadata: [WinError 10053] An established connection was aborted by the software in your host machine
Traceback (most recent call last):
  File "c:\Users\dominik.mika\dp100\dp100-learn\venv\Lib\site-packages\opentelemetry\resource\detector\azure\vm.py", line 66, in _get_azure_vm_metadata
    with urlopen(request, timeout=0.2) as response:
         ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dominik.mika\AppData\Local\Programs\Python\Python313\Lib\urllib\request.py", line 189, in urlopen
    return opener.open(url, data, timeout)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\dominik.mika\dp100\dp100-learn\venv\Lib\site-packages\opentelemetry\instrumentation\urllib\__init__.py", line 242, in instrumented_open
    return _instrumented_open_call(
        opener, request_, call_wrapped, get_or_create_headers
    )
  File "c:\Users\dominik.mika\dp100\dp100-learn\venv\Lib\site-packages\opentelemetry\instrumentation\urllib\__init__.py", line 250, in 

AML workspace already exists: mlw-dp100-labs


## Setup the Data

## Create a DataStore

In [38]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration

store = AzureBlobDatastore(
    name=datastore_name,
    description="Blob Storage for DP-100 certification prep",
    account_name=storage_account_name,
    container_name=storage_container_name, 
    credentials=AccountKeyConfiguration(
        account_key=storage_account_access_key
    ),
    type="azure_blob",
)

ml_client.create_or_update(store)

AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'dmdp100datastore', 'description': 'Blob Storage for DP-100 certification prep', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/datastores/dmdp100datastore', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dominik.mika\\dp100\\dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x00000144F2D6A580>, 'credentials': {'type': 'account_key'}, 'container_name': 'dmpdp100data', 'account_name': 'dmpdp100storageaccount99', 'endpoint': 'core.windows.net', 'protocol': 'https'})

In [39]:
stores = ml_client.datastores.list()
for ds_name in stores:
    print(ds_name.name)

dmdp100datastore
azureml_globaldatasets
workspaceblobstore
workspacefilestore
workspaceartifactstore
workspaceworkingdirectory


## Create Data Assets

In [52]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_path = './data/telco-churn-data/telco-customer-churn.csv'

my_data = Data(
    path=file_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FILE,
    description="Data asset pointing to a local file, automatically uploaded to the default datastore",
    name="telco-churn-file"
)

ml_client.data.create_or_update(my_data)

Uploading telco-customer-churn.csv (< 1 MB): 0.00B [00:00, ?B/s] (< 1 MB): 100%|##########| 978k/978k [00:00<00:00, 3.84MB/s] (< 1 MB): 100%|##########| 978k/978k [00:00<00:00, 3.75MB/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/211bf8d0be7de258b660a998ca7be5af/telco-customer-churn.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-file', 'description': 'Data asset pointing to a local file, automatically uploaded to the default datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-file/versions/2', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dominik.mika\\dp100\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x00000144F2A41D90>, 'serializ

In [51]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder_path = './data/telco-churn-data'

my_data = Data(
    path=folder_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FOLDER,
    description="Data asset pointing to data-asset-path folder in datastore",
    name="telco-churn-folder",
)

ml_client.data.create_or_update(my_data)

Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/bd39861715370fa5325c0653da11de64/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-folder', 'description': 'Data asset pointing to data-asset-path folder in datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-folder/versions/2', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dominik.mika\\dp100\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x00000144F2D2D6E0>, 'serialize': <msrest.serialization.S

In [45]:
%%writefile data/telco-churn-data/MLTable

paths:
  - file: ./data/telco-churn-data/telco-customer-churn.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'ascii'

Writing data/telco-churn-data/MLTable


In [50]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path = './data/telco-churn-data'

my_data = Data(
    path=data_path,
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to telco-customer-churn.csv in data folder",
    name="telco-churn-table",
    datastore=datastore_name,

)

ml_client.data.create_or_update(my_data)

Uploading telco-churn-data (0.98 MBs): 100%|##########| 977661/977661 [00:00<00:00, 1726359.30it/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/bd39861715370fa5325c0653da11de64/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./data/telco-churn-data/telco-customer-churn.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-table', 'description': 'MLTable pointing to telco-customer-churn.csv in data folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-table/versions/2', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dominik.mika\\dp100\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x00000144F2D

## Read the Data Assets

In [16]:
data_asset = ml_client.data.get("telco-churn-file", version="1")
df = pd.read_csv(data_asset.path)
df

ValueError: Protocol not known: azureml

In [14]:
data_asset.id

'/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-file/versions/2'

In [15]:
df = pd.read_csv(data_asset.id)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-file/versions/2'

## Setup an Environment

## Setup the Compute

### Compute Cluster

### Compute Instance

### Create a datastore